<h2>Lab 3 Pipeline: NYC Taxi Analysis</h2>

In [1]:
from dask.distributed import LocalCluster

In [2]:
import dask
import distributed
import msgpack

print("dask:", dask.__version__)
print("distributed:", distributed.__version__)
print("msgpack:", msgpack.__version__)

dask: 2025.1.0
distributed: 2025.1.0
msgpack: 1.1.2


In [3]:
from dask.distributed import Client

client = Client("tcp://localhost:8786")
client

C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:1612: VersionMismatchWarning: Mismatched versions found

+-------------+----------------+-----------------+-----------------+
| Package     | Client         | Scheduler       | Workers         |
+-------------+----------------+-----------------+-----------------+
| cloudpickle | 3.0.0          | 3.1.1           | 3.1.1           |
| lz4         | 4.3.2          | 4.3.3           | 4.3.3           |
| msgpack     | 1.1.2          | 1.1.0           | 1.1.0           |
| numpy       | 2.1.3          | 2.2.2           | 2.2.2           |
| python      | 3.13.5.final.0 | 3.10.12.final.0 | 3.10.12.final.0 |
| toolz       | 1.0.0          | 0.12.0          | 0.12.0          |
| tornado     | 6.5.1          | 6.4.2           | 6.4.2           |
+-------------+----------------+-----------------+-----------------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


<Client: 'tcp://10.244.0.6:8786' processes=2 threads=8, memory=15.51 GiB>

In [4]:
import os
import time
import urllib.request
import pandas as pd
import dask.dataframe as dd


DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
LOCAL_PATH = "data/yellow_tripdata_2024-01.parquet"


def download_data(url, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f"Downloading {url}...")
        urllib.request.urlretrieve(url, dest)
    print(f"Data ready: {dest} ({os.path.getsize(dest) / 1e6:.1f} MB)")
    return dest


def run_pipeline(client, ddf):
    start = time.time()

    print(f"Connected to: {client}")
    print(f"Workers: {len(client.scheduler_info()['workers'])}")
    print("\nRunning pipeline...")

    result_1 = (
        ddf.groupby("PULocationID")["trip_distance"]
        .mean()
        .compute()
        .sort_values(ascending=False)
        .head(10)
    )

    result_2 = (
        ddf.groupby("payment_type")["total_amount"]
        .mean()
        .compute()
    )

    result_3 = (
        ddf[ddf["passenger_count"] > 0]
        .groupby("passenger_count")["tip_amount"]
        .mean()
        .compute()
    )

    elapsed = time.time() - start

    print("\nTop 10 average trip distance by pickup location:")
    print(result_1)

    print("\nAverage total amount by payment type:")
    print(result_2)

    print("\nAverage tip amount by passenger count:")
    print(result_3)

    print(f"\nPipeline completed in {elapsed:.1f} seconds")
    return elapsed

In [5]:
download_data(DATA_URL, LOCAL_PATH)

Data ready: data/yellow_tripdata_2024-01.parquet (50.0 MB)


'data/yellow_tripdata_2024-01.parquet'

In [8]:
import pandas as pd
import dask.dataframe as dd

pdf = pd.read_parquet(LOCAL_PATH)
ddf = dd.from_pandas(pdf, npartitions=12)

deployed_time = run_pipeline(client, ddf)

Connected to: <Client: 'tcp://10.244.0.6:8786' processes=2 threads=8, memory=15.51 GiB>
Workers: 2

Running pipeline...


C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 33.93 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 45.24 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 45.25 MiB.
This may cause some slowdown.
Consider loading the data with Das


Top 10 average trip distance by pickup location:
PULocationID
40     234.680787
33      62.022951
66      58.673560
44      57.200000
4       29.954406
204     29.580000
209     20.233609
244     19.434363
151     19.001850
172     18.060000
Name: trip_distance, dtype: float64

Average total amount by payment type:
payment_type
1    28.258861
2    22.884506
3     8.755475
4     1.773829
0    25.811737
Name: total_amount, dtype: float64

Average tip amount by passenger count:
passenger_count
1.0     3.371262
2.0     3.717130
3.0     3.537046
4.0     3.466038
5.0     3.379708
6.0     3.344099
7.0     8.370000
8.0    11.972157
9.0     3.050000
Name: tip_amount, dtype: float64

Pipeline completed in 15.0 seconds


In [9]:
import pandas as pd
import dask.dataframe as dd

pdf = pd.read_parquet(LOCAL_PATH)
ddf = dd.from_pandas(pdf, npartitions=12)

deployed_time = run_pipeline(client, ddf)

Connected to: <Client: 'tcp://10.244.0.6:8786' processes=2 threads=8, memory=15.51 GiB>
Workers: 2

Running pipeline...


C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 33.93 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 45.24 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 45.25 MiB.
This may cause some slowdown.
Consider loading the data with Das


Top 10 average trip distance by pickup location:
PULocationID
40     234.680787
33      62.022951
66      58.673560
44      57.200000
4       29.954406
204     29.580000
209     20.233609
244     19.434363
151     19.001850
172     18.060000
Name: trip_distance, dtype: float64

Average total amount by payment type:
payment_type
1    28.258861
2    22.884506
3     8.755475
4     1.773829
0    25.811737
Name: total_amount, dtype: float64

Average tip amount by passenger count:
passenger_count
1.0     3.371262
2.0     3.717130
3.0     3.537046
4.0     3.466038
5.0     3.379708
6.0     3.344099
7.0     8.370000
8.0    11.972157
9.0     3.050000
Name: tip_amount, dtype: float64

Pipeline completed in 15.3 seconds


In [10]:
print(f"Deployed cluster execution time: {deployed_time:.1f} seconds")
print(f"Connected workers: {len(client.scheduler_info()['workers'])}")

Deployed cluster execution time: 15.3 seconds
Connected workers: 2


In [11]:
from dask.distributed import LocalCluster, Client

with LocalCluster(n_workers=2, threads_per_worker=2, memory_limit="2GB") as cluster:
    with Client(cluster) as local_client:
        pdf_local = pd.read_parquet(LOCAL_PATH)
        ddf_local = dd.from_pandas(pdf_local, npartitions=12)
        local_time = run_pipeline(local_client, ddf_local)

print(f"Local execution time: {local_time:.1f} seconds")
print(f"Deployed execution time: {deployed_time:.1f} seconds")

Connected to: <Client: 'tcp://127.0.0.1:53200' processes=2 threads=4, memory=3.73 GiB>
Workers: 2

Running pipeline...


C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 33.93 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 45.24 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
C:\Users\User\anaconda3\Lib\site-packages\distributed\client.py:3370: UserWarning: Sending large graph of size 45.25 MiB.
This may cause some slowdown.
Consider loading the data with Das


Top 10 average trip distance by pickup location:
PULocationID
40     234.680787
33      62.022951
66      58.673560
44      57.200000
4       29.954406
204     29.580000
209     20.233609
244     19.434363
151     19.001850
172     18.060000
Name: trip_distance, dtype: float64

Average total amount by payment type:
payment_type
1    28.258861
2    22.884506
3     8.755475
4     1.773829
0    25.811737
Name: total_amount, dtype: float64

Average tip amount by passenger count:
passenger_count
1.0     3.371262
2.0     3.717130
3.0     3.537046
4.0     3.466038
5.0     3.379708
6.0     3.344099
7.0     8.370000
8.0    11.972157
9.0     3.050000
Name: tip_amount, dtype: float64

Pipeline completed in 6.6 seconds
Local execution time: 6.6 seconds
Deployed execution time: 15.3 seconds


In [ ]:
client.close()
print("Client connection closed.")

In [ ]:
helm uninstall my-dask
minikube stop